# Shortest Paths and Dijkstra


```{contents}
:local:
:depth: 2
```


```{index} Improve Tentative Routes Before Declaring Them Final
```



```{index} Improve Tentative Routes Before Declaring Them Final
```

## Improve Tentative Routes Before Declaring Them Final



Routing asks the least cost from one source to each reachable destination. A tentative distance represents the best discovered route so far. Relaxing arc u-to-v compares the existing tentative value for v with distance[u] plus the arc cost. Replace it only when the new route is strictly better, and record u as predecessor.

Use directed arcs A-to-B cost 10, A-to-C cost 1, and C-to-B cost 1. Starting at A, B is first discovered at cost ten. Processing C improves B to cost two. A priority-queue entry for the old ten-cost B may still exist, so the implementation must distinguish current best distance from stale queued proposals.

| Freshly processed vertex | Best A | Best B | Best C |
| --- | --- | --- | --- |
| A at 0 | 0 | 10 | 1 |
| C at 1 | 0 | 2 | 1 |
| B at 2 | 0 | 2 | 1 |

With nonnegative arc weights, a fresh minimum tentative distance can be settled: an undiscovered cheaper route would need to pass through a frontier prefix no more expensive than that route, contradicting selection of the minimum. A negative later edge breaks this argument. Do not generalize from one negative-edge example that happens to return the right result.

When using duplicate queue entries, compare a removed entry's priority with the authoritative best distance. Skip stale entries rather than rescanning their neighbors as if the old route were current. Strict improvement avoids producing repeated equal-distance updates around zero-weight cycles. Tie ordering may choose different equally short predecessor paths without changing shortest distances.

For related explanation, see the further reading below.[^development-reading]


```{index} Check the Witness as Well as the Number
```



```{index} Check the Witness as Well as the Number
```

## Check the Witness as Well as the Number



A distance of two for B should have a path witness A-C-B whose directed arcs exist and sum to two. A known unreachable destination has no such path, while the source has a one-vertex zero-cost path to itself. Unknown labels, unreachable known labels and arithmetic overflow are distinct cases with explicit policies in the existing implementation.

Guided counterexample: in the earlier undirected triangle, the cheapest cable network used A-B and B-C, total four. But the cheapest A-to-C route uses the direct edge of cost three rather than the network path of cost four. Explain why both results can be optimal for their respective objectives. A minimum spanning tree and a shortest-path tree are not interchangeable merely because both are trees with weighted edges.


```{index} Dijkstra with Nonnegative Directed Arcs
```

## Dijkstra with Nonnegative Directed Arcs




```{index} Dijkstra with Nonnegative Directed Arcs
```

Dijkstra minimizes path cost from one known source. Directed arcs here specify allowed travel direction; supply both directions when modeling two-way roads. Validate every arc, including unreachable ones, before running: weights must be nonnegative, endpoints known, and loops excluded by this teaching interface. Zero weights are valid. Nullable long distances represent infinity without confusing a valid int.MaxValue path with absence.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"HQ",Console.WriteLine);
foreach(string v in vertices)Console.WriteLine($"HQ->{v}: {(routes.Distance[v]?.ToString()??"unreachable")}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


settle HQ (0)
relax HQ->Shop: 2
relax HQ->Lab: 6
settle Shop (2)
relax Shop->Lab: 5
relax Shop->Depot: 7
settle Lab (5)
skip stale Lab (6)
settle Depot (7)
HQ->HQ: 0
HQ->Lab: 5
HQ->Shop: 2
HQ->Depot: 7
HQ->Remote: unreachable


Relaxation replaces a tentative distance only on strict improvement. Each finite tentative distance corresponds to a discovered source route. A fresh minimum-priority pop has the final shortest distance: any hypothetical shorter route would first leave the already settled set with a prefix no longer than that route; nonnegative remaining weights cannot make a later prefix cheaper. Zero weights preserve the argument. Negative edges invalidate this reasoning, even if a particular example happens to produce correct output.

**Visualize it.** OpenDSA's [Single-Source Shortest Paths](https://opendsa-server.cs.vt.edu/OpenDSA/Books/CS3/html/GraphShortest.html#single-source-shortest-paths) slideshow animates Dijkstra's tentative distances being improved and then settled one vertex at a time.

**Common mistake.** Supplying one arc per road for a two-way network. Dijkstra then finds routes in one direction only, and `Remote` is not the only unreachable vertex; add both directions explicitly, as the triangle example does.


```{index} Stale Queue Entries and Qualified Costs
```

## Stale Queue Entries and Qualified Costs




```{index} Stale Queue Entries and Qualified Costs
```

Improving a distance enqueues a new entry; the old entry remains in the library queue. Compare its stored priority with the current best distance and skip stale entries. Strict improvement also avoids endless equal-distance updates around zero-weight cycles.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"A","B","C","D","Isolate"};
var arcs=new[]{new GreedyGraph.Edge("A","B",10),new GreedyGraph.Edge("A","C",1),new GreedyGraph.Edge("C","B",1),new GreedyGraph.Edge("B","D",0),new GreedyGraph.Edge("D","B",0)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"A",Console.WriteLine);
Console.WriteLine($"D distance={routes.Distance["D"]}, isolate={(routes.Distance["Isolate"] is null)}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


settle A (0)
relax A->B: 10
relax A->C: 1
settle C (1)
relax C->B: 2
settle B (2)
relax B->D: 2
settle D (2)
skip stale B (10)
D distance=2, isolate=True


B first enters the queue at ten, then improves to two. Its old ten-priority entry is skipped. Each reachable adjacency list is scanned on its fresh pop; at most E successful relaxations enqueue entries. This lazy-duplicate implementation can hold O(E) queue entries and has O(V+E log(E+1)) time for nontrivial graphs, plus O(V+E) storage. An indexed decrease-key implementation has different queue/storage bounds. Equal priorities need not have stable library ordering; distances are correct and the chosen shortest predecessor route may vary on ties.

**Common mistake.** Relaxing with `<=` instead of `<`. On the zero-weight cycle between B and D every pop re-enqueues the other vertex at an equal distance, and the loop runs until the queue is full of duplicates.


```{index} Reconstruct Only Reachable Routes
```

## Reconstruct Only Reachable Routes




```{index} Reconstruct Only Reachable Routes
```

A predecessor records the previous vertex on a strict improving route. Follow predecessors backward and reverse with a stack. Return an empty path for a known unreachable vertex, a singleton for the source, and reject an unknown target. Distance/predecessor maps are read-only views; do not fabricate a one-vertex route for an unreachable destination.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"HQ");
foreach(string target in new[]{"HQ","Depot","Remote"})
 Console.WriteLine($"{target}: [{string.Join(" -> ",routes.PathTo(target))}]");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


HQ: [HQ]
Depot: [HQ -> Shop -> Depot]
Remote: []


Every returned route must begin at the source, end at its target, use directed arcs, and sum to the reported distance. Route reconstruction costs O(path length). Long accumulation avoids int overflow; checked arithmetic rejects overflow rather than silently wrapping. With int weights and finite simple paths in a practical array-sized graph, long provides ample range; its policy is still explicit.

**Common mistake.** Returning `[target]` for an unreachable vertex because the loop that follows predecessors found none. A one-vertex route reads as "the target is the source"; the method must check the distance first and return an empty path.


```{index} Shortest Paths and String Matching
```

<span id="shortest-paths-and-string-matching"></span>


Algorithm assumptions determine what an answer means. BFS minimizes hops, Dijkstra minimizes nonnegative total weights, and a string matcher needs a precise equality and occurrence policy. Reusing justified information can eliminate repeated work without changing the contract.

```{index} Minimum Hops Is Not Minimum Cost
```

## Minimum Hops Is Not Minimum Cost




```{index} Minimum Hops Is Not Minimum Cost
```

Suppose HQ→Client has one edge costing 100, while HQ→Helpdesk→Client costs 1+1. BFS chooses one hop; a nonnegative weighted shortest-path algorithm chooses cost two. Chapter 24 provides the complete guarded Dijkstra implementation. Negative weights require another algorithm, and a reachable negative cycle on a route to the target can mean no finite minimum exists.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;
using System.Numerics;

var graph=new GraphWalk(new Dictionary<string,string[]>{["HQ"]=new[]{"Client","Helpdesk"},["Helpdesk"]=new[]{"Client"},["Client"]=Array.Empty<string>()});
var bfs=graph.Bfs("HQ");
Console.WriteLine($"minimum-hop route={string.Join("->",bfs.PathTo("Client"))}, hops={bfs.Distance["Client"]}");
Console.WriteLine("that route costs 100; feasible two-hop route costs 2");
public sealed class GraphWalk
{
    private readonly Dictionary<string,string[]> graph;
    private readonly string[] vertices;
    public GraphWalk(Dictionary<string,string[]> adjacency)
    {
        ArgumentNullException.ThrowIfNull(adjacency);
        graph=new Dictionary<string,string[]>(StringComparer.Ordinal);
        foreach(var pair in adjacency)
        {
            if(pair.Value is null)throw new ArgumentException("Null adjacency list.");
            graph.Add(pair.Key,pair.Value.Distinct(StringComparer.Ordinal).ToArray());
        }
        foreach(var neighbors in graph.Values)
            if(neighbors.Any(v=>v is null||!graph.ContainsKey(v)))throw new ArgumentException("Unknown endpoint.");
        vertices=graph.Keys.OrderBy(v=>v,StringComparer.Ordinal).ToArray();
    }
    public sealed class Layers
    {
        public string Source { get; }
        public string[] Order { get; }
        public IReadOnlyDictionary<string,int?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        public long EdgesExamined { get; }
        internal Layers(string source,string[] order,Dictionary<string,int?> distance,Dictionary<string,string?> previous,long edges)
        {
            Source=source;Order=order;EdgesExamined=edges;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,int?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            while(at is not null)
            {
                if(path.Count>=Distance.Count)throw new InvalidOperationException("Predecessor cycle.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Invalid source path.");
            return path.ToArray();
        }
    }
    public Layers Bfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var distance=vertices.ToDictionary(v=>v,_=>(int?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new Queue<string>();var order=new List<string>();long edges=0;
        distance[source]=0;queue.Enqueue(source);
        while(queue.Count>0)
        {
            string at=queue.Dequeue();order.Add(at);trace?.Invoke($"visit {at}: layer={distance[at]}");
            foreach(string next in graph[at])
            {
                edges++;
                if(distance[next] is not null)continue;
                distance[next]=distance[at]!.Value+1;previous[next]=at;queue.Enqueue(next);
            }
        }
        return new Layers(source,order.ToArray(),distance,previous,edges);
    }
    public record Depth(string[] Preorder,string[] FinishOrder,long EdgesExamined);
    public Depth Dfs(string source,Action<string>? trace=null)
    {
        if(!graph.ContainsKey(source))throw new KeyNotFoundException(source);
        var color=vertices.ToDictionary(v=>v,_=>0,StringComparer.Ordinal);
        var stack=new Stack<(string Vertex,int Next)>();var pre=new List<string>();var finish=new List<string>();long edges=0;
        Enter(source);
        while(stack.Count>0)
        {
            var frame=stack.Pop();
            if(frame.Next==graph[frame.Vertex].Length)
            {color[frame.Vertex]=2;finish.Add(frame.Vertex);trace?.Invoke($"exit {frame.Vertex}");continue;}
            stack.Push((frame.Vertex,frame.Next+1));string next=graph[frame.Vertex][frame.Next];edges++;
            if(color[next]==0)Enter(next);
            else if(color[next]==1)trace?.Invoke($"active edge {frame.Vertex}->{next}");
        }
        return new Depth(pre.ToArray(),finish.ToArray(),edges);
        void Enter(string vertex){color[vertex]=1;pre.Add(vertex);stack.Push((vertex,0));trace?.Invoke($"enter {vertex}");}
    }
    public string[][] UndirectedComponents()
    {
        var neighbors=graph.ToDictionary(p=>p.Key,p=>p.Value.ToHashSet(StringComparer.Ordinal),StringComparer.Ordinal);
        foreach(var pair in graph)
            foreach(string next in pair.Value)
                if(!neighbors[next].Contains(pair.Key))throw new ArgumentException("Components require symmetric adjacency.");
        var seen=new HashSet<string>(StringComparer.Ordinal);var groups=new List<string[]>();
        foreach(string root in vertices)
        {
            if(!seen.Add(root))continue;
            var queue=new Queue<string>();var group=new List<string>();queue.Enqueue(root);
            while(queue.Count>0)
            {
                string at=queue.Dequeue();group.Add(at);
                foreach(string next in graph[at])if(seen.Add(next))queue.Enqueue(next);
            }
            groups.Add(group.ToArray());
        }
        return groups.ToArray();
    }
}


minimum-hop route=HQ->Client, hops=1
that route costs 100; feasible two-hop route costs 2


This is a verified objective counterexample, not a reason to call BFS incorrect. Declare units, direction, permitted weights, and whether the requested result is reachability, edge count, total cost, or all destinations before selecting an algorithm.

```{index} The Cheapest Network Is Not a Shortest-Route Tree
```

## The Cheapest Network Is Not a Shortest-Route Tree




```{index} The Cheapest Network Is Not a Shortest-Route Tree
```

On an undirected triangle, A–B=2, B–C=2, A–C=3. The MST has total cost four, but its A-to-C route costs four; the direct road costs three. Thus minimal network total does not imply minimal source routes.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"A","B","C"};
var edges=new[]{new GreedyGraph.Edge("A","B",2),new GreedyGraph.Edge("B","C",2),new GreedyGraph.Edge("A","C",3)};
var forest=GreedyGraph.Kruskal(vertices,edges);
var arcs=edges.Concat(edges.Select(e=>new GreedyGraph.Edge(e.To,e.From,e.Weight))).ToArray();
var routes=GreedyGraph.Dijkstra(vertices,arcs,"A");
Console.WriteLine($"MST total={forest.Cost}, MST A-C route=4, shortest A-C={routes.Distance["C"]}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


MST total=4, MST A-C route=4, shortest A-C=3


Choose the algorithm by the objective and graph assumptions, not keywords in a scenario string. Directed road routing is not an undirected MST problem. Neither model automatically solves a tour visiting every store once or a network with resilience/capacity constraints.

**Common mistake.** Running Kruskal because the scenario says "cheapest" and then reading routes off the result. The cheapest network minimizes total cable, and the route A to C along it costs four against a direct road of three.


```{index} Exercise
```

## Exercise




```{index} Exercise
```

Verify a distance larger than int.MaxValue, a zero edge, and an unreachable vertex. Print route and distance; reject a negative arc even in a disconnected component.

This starter includes its supporting types and operations below the marked client area. Run it from a fresh session without running a previous demonstration or revealing the answer. Your task is to write the requested calls, assertions and interpretation, not to recreate the implementation. Compare the ordinary result and each boundary result against the contract before inspecting the hidden solution.


In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"A","B","C","Remote"};
var arcs=new[]{new GreedyGraph.Edge("A","B",int.MaxValue),new GreedyGraph.Edge("B","C",1)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.
// Cell-local support: inspect the operations, then write the client checks above.
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
string[] vertices={"A","B","C","Remote"};
var arcs=new[]{new GreedyGraph.Edge("A","B",int.MaxValue),new GreedyGraph.Edge("B","C",1)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"A");
Console.WriteLine($"C={routes.Distance["C"]}, route={string.Join("->",routes.PathTo("C"))}, remote path length={routes.PathTo("Remote").Length}");
try{GreedyGraph.Dijkstra(vertices,arcs.Append(new GreedyGraph.Edge("Remote","C",-1)).ToArray(),"A");}catch(ArgumentOutOfRangeException){Console.WriteLine("unreachable negative arc rejected");}
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


C=2147483648, route=A->B->C, remote path length=0
unreachable negative arc rejected


The output is `C=2147483648, route=A->B->C, remote path length=0` and `unreachable negative arc rejected`. The distance exceeds `int.MaxValue` by one and is correct because distances are `long`; the unreachable vertex gets an empty path; and the negative arc is rejected by validation even though no route would ever use it.


```{index} Verify Delivery Routes Independently
```

## Verify Delivery Routes Independently




```{index} Verify Delivery Routes Independently
```

Supply directed travel arcs and check each reconstructed path’s cost against the reported distance. Check source, unreachable, and reachable targets. A reachable path must not be accepted merely because its endpoint matches.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var routes=GreedyGraph.Dijkstra(vertices,arcs,"HQ");
foreach(string target in vertices)
{
 string[] path=routes.PathTo(target);
 if(routes.Distance[target] is null){if(path.Length!=0)throw new Exception("Unreachable path.");Console.WriteLine($"{target}: unreachable");continue;}
 if(path[0]!="HQ"||path[^1]!=target)throw new Exception("Path endpoints.");
 long cost=0;
 for(int i=1;i<path.Length;i++)cost+=arcs.Where(e=>e.From==path[i-1]&&e.To==path[i]).Min(e=>e.Weight);
 if(cost!=routes.Distance[target])throw new Exception("Path cost mismatch.");
 Console.WriteLine($"{target}: {cost}, {string.Join("->",path)}");
}
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


HQ: 0, HQ
Lab: 5, HQ->Shop->Lab
Shop: 2, HQ->Shop
Depot: 7, HQ->Shop->Depot
Remote: unreachable


The test scans arcs to validate the path, so its test-oracle cost is different from production reconstruction. Parallel arcs are allowed; a shortest predecessor step uses the cheapest usable arc between that pair under this representation. Path tests complement distance-reference checks.

**Common mistake.** Accepting a path because its first element is the source and its last is the target. The check must also confirm that each consecutive pair is a stored arc and that the arc weights sum to the reported distance.


```{index} Boundary Policies Are Part of Correctness
```

## Boundary Policies Are Part of Correctness




```{index} Boundary Policies Are Part of Correctness
```

Exercise empty forests, singleton/isolate routes, zero weights, signed MST edges, and invalid inputs. MST negative costs are valid; Dijkstra negative costs are not. Distinguish invalid inputs from valid but disconnected data.

In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

var empty=GreedyGraph.Kruskal(Array.Empty<string>(),Array.Empty<GreedyGraph.Edge>());
Console.WriteLine($"empty forest: cost={empty.Cost}, components={empty.Components}");
var forest=GreedyGraph.Kruskal(new[]{"A","B"},new[]{new GreedyGraph.Edge("A","B",-2)});
Console.WriteLine($"negative MST cost={forest.Cost}");
var singleton=GreedyGraph.Dijkstra(new[]{"A"},Array.Empty<GreedyGraph.Edge>(),"A");
Console.WriteLine($"singleton path={string.Join("->",singleton.PathTo("A"))}");
try{GreedyGraph.Kruskal(new[]{"A"},new[]{new GreedyGraph.Edge("A","Missing",1)});}catch(ArgumentException){Console.WriteLine("unknown endpoint rejected");}
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


empty forest: cost=0, components=0
negative MST cost=-2
singleton path=A
unknown endpoint rejected


The output is `empty forest: cost=0, components=0`, `negative MST cost=-2`, `singleton path=A`, and `unknown endpoint rejected`. Each line is a boundary the contract names: an empty input is valid, a negative cable cost is valid for a forest, a source's route to itself is one vertex, and a reference to a vertex that was never added is an error rather than an isolate.

**Common mistake.** Rejecting negative weights everywhere. A negative cable cost is valid for a spanning forest, where edges are chosen rather than traversed; only Dijkstra's argument fails on negative arcs.


```{index} Choose the Model Before the Rule
```

## Choose the Model Before the Rule




```{index} Choose the Model Before the Rule
```

Write down the objective, units, direction, required connectivity, and weight assumptions. Scheduling maximizes compatible count; MST minimizes network total; Dijkstra minimizes route cost from a source. Preserve the feasibility and optimality arguments separately.

A facilities decision can need all three models, but their numeric costs are not interchangeable. Cable cost can be currency, road cost travel time, and meeting objective a count. More realistic constraints may require a different algorithm entirely.

```{index} Exercise
```

## Practice: Compare Facilities Models




```{index} Exercise
```

Add a direct HQ-to-Depot travel arc of cost one and compare routing with the unchanged cable forest. Explain why a route improvement need not alter the separate cable-input optimum.

This starter includes its supporting types and operations below the marked client area. Run it from a fresh session without running a previous demonstration or revealing the answer. Your task is to write the requested calls, assertions and interpretation, not to recreate the implementation. Compare the ordinary result and each boundary result against the contract before inspecting the hidden solution.


In [ ]:
using System;
using System.Linq;
using System.Collections.Generic;

string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
// Your code starts here.
/* TODO: complete the task. */
// Your code ends here.
// Cell-local support: inspect the operations, then write the client checks above.
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


In [1]:
using System;
using System.Linq;
using System.Collections.Generic;

// Solution
string[] vertices={"HQ","Lab","Shop","Depot","Remote"};
var edges=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var arcs=new[]{new GreedyGraph.Edge("HQ","Shop",2),new GreedyGraph.Edge("HQ","Lab",6),new GreedyGraph.Edge("Shop","Lab",3),new GreedyGraph.Edge("Lab","Depot",4),new GreedyGraph.Edge("Shop","Depot",5)};
var forest=GreedyGraph.Kruskal(vertices,edges);
var routes=GreedyGraph.Dijkstra(vertices,arcs.Append(new GreedyGraph.Edge("HQ","Depot",1)).ToArray(),"HQ");
Console.WriteLine($"cable forest={forest.Cost}, delivery Depot={routes.Distance["Depot"]}");
public static class GreedyGraph
{
    public record Edge(string From,string To,int Weight);
    public record Forest(Edge[] Edges,long Cost,int Components);
    public sealed class Routes
    {
        public string Source { get; }
        public IReadOnlyDictionary<string,long?> Distance { get; }
        public IReadOnlyDictionary<string,string?> Previous { get; }
        internal Routes(string source,Dictionary<string,long?> distance,Dictionary<string,string?> previous)
        {
            Source=source;
            Distance=new System.Collections.ObjectModel.ReadOnlyDictionary<string,long?>(distance);
            Previous=new System.Collections.ObjectModel.ReadOnlyDictionary<string,string?>(previous);
        }
        public string[] PathTo(string target)
        {
            if(!Distance.ContainsKey(target))throw new KeyNotFoundException(target);
            if(Distance[target] is null)return Array.Empty<string>();
            var path=new Stack<string>();string? at=target;
            for(int steps=0;at is not null;steps++)
            {
                if(steps>=Distance.Count)throw new InvalidOperationException("Invalid predecessor chain.");
                path.Push(at);at=Previous[at];
            }
            if(path.Peek()!=Source)throw new InvalidOperationException("Path does not reach source.");
            return path.ToArray();
        }
    }
    // Undirected edges; parallel edges allowed. Negative MST weights are valid.
    public static Forest Kruskal(string[] vertices,Edge[] edges,Action<string>? trace=null)
    {
        Validate(vertices,edges,false);
        var parent=vertices.ToDictionary(v=>v,v=>v,StringComparer.Ordinal);
        var size=vertices.ToDictionary(v=>v,_=>1,StringComparer.Ordinal);
        int components=vertices.Length;long cost=0;var chosen=new List<Edge>();
        foreach(var edge in edges.OrderBy(e=>e.Weight).ThenBy(e=>e.From,StringComparer.Ordinal).ThenBy(e=>e.To,StringComparer.Ordinal))
        {
            string a=Find(edge.From),b=Find(edge.To);
            if(a==b){trace?.Invoke($"reject {edge.From}-{edge.To} ({edge.Weight}): cycle");continue;}
            if(size[a]<size[b])(a,b)=(b,a);
            parent[b]=a;size[a]+=size[b];components--;
            chosen.Add(edge);cost=checked(cost+edge.Weight);
            trace?.Invoke($"accept {edge.From}-{edge.To} ({edge.Weight}): components={components}");
        }
        return new Forest(chosen.ToArray(),cost,components);
        string Find(string vertex)
        {
            while(parent[vertex]!=vertex)
            {parent[vertex]=parent[parent[vertex]];vertex=parent[vertex];}
            return vertex;
        }
    }
    // Directed arcs; nonnegative weights required, including unreachable arcs.
    public static Routes Dijkstra(string[] vertices,Edge[] arcs,string source,Action<string>? trace=null)
    {
        Validate(vertices,arcs,true);
        var graph=vertices.ToDictionary(v=>v,_=>new List<Edge>(),StringComparer.Ordinal);
        if(!graph.ContainsKey(source))throw new KeyNotFoundException("Unknown source.");
        foreach(var edge in arcs)graph[edge.From].Add(edge);
        var distance=vertices.ToDictionary(v=>v,_=>(long?)null,StringComparer.Ordinal);
        var previous=vertices.ToDictionary(v=>v,_=>(string?)null,StringComparer.Ordinal);
        var queue=new PriorityQueue<string,long>();distance[source]=0;queue.Enqueue(source,0);
        while(queue.TryDequeue(out string? current,out long queued))
        {
            if(distance[current]!=queued){trace?.Invoke($"skip stale {current} ({queued})");continue;}
            trace?.Invoke($"settle {current} ({queued})");
            foreach(var edge in graph[current])
            {
                long candidate=checked(queued+edge.Weight);
                if(distance[edge.To] is null || candidate<distance[edge.To])
                {
                    distance[edge.To]=candidate;previous[edge.To]=current;
                    queue.Enqueue(edge.To,candidate);
                    trace?.Invoke($"relax {current}->{edge.To}: {candidate}");
                }
            }
        }
        return new Routes(source,distance,previous);
    }
    private static void Validate(string[] vertices,Edge[] edges,bool nonnegative)
    {
        ArgumentNullException.ThrowIfNull(vertices);ArgumentNullException.ThrowIfNull(edges);
        var known=new HashSet<string>(StringComparer.Ordinal);
        foreach(string vertex in vertices)
            if(vertex is null||!known.Add(vertex))throw new ArgumentException("Vertices must be distinct non-null labels.");
        foreach(var edge in edges)
        {
            if(edge is null||!known.Contains(edge.From)||!known.Contains(edge.To))throw new ArgumentException("Unknown endpoint.");
            if(edge.From==edge.To)throw new ArgumentException("This graph contract excludes self-loops.");
            if(nonnegative&&edge.Weight<0)throw new ArgumentOutOfRangeException(nameof(edges),"Dijkstra requires nonnegative weights.");
        }
    }
}


cable forest=9, delivery Depot=1


The output is `cable forest=9, delivery Depot=1`. The new one-minute arc made the delivery route to Depot cheaper, but the cable forest was computed from the original undirected edges and still costs 9: the two answers come from two inputs and two objectives, so one can change without the other.


```{rubric} Footnotes
```


[^2402_spanning_trees_shortest_paths-1]: Union-find’s near-constant amortized bound assumes union by size/rank and path compression; the inverse Ackermann factor α grows extremely slowly.
[^2402_spanning_trees_shortest_paths-2]: Dijkstra requires nonnegative edges. Algorithms for negative edges, including Bellman–Ford, belong in a later course; a validation failure is not a reason to silently run Dijkstra anyway.

[^2403_optimization_lab-1]: Small exhaustive or independent reference checks detect implementation mistakes without replacing the general proof sketch.
[^2403_optimization_lab-2]: The cable and delivery examples use separate edge sets and units. Do not silently treat a directed road as an undirected cable.

[^development-reading]: Further reading: [Sedgewick and Wayne, Minimum Spanning Trees](https://algs4.cs.princeton.edu/43mst/index.php). The worked business scenarios and guided reasoning here are original teaching examples.
